# Students Performance in Exams - EDA Analysis

The dataset of this notebook is provided by Kaggle : https://www.kaggle.com/spscientist/students-performance-in-exams. 

# Brief description of dataset

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline
plt.style.use('ggplot')

Reading data

In [ ]:
df = pd.read_csv('data/StudentsPerformance.csv')

In [ ]:
df.head(n = 5)

In [ ]:
df.info()

In [ ]:
df.shape

Modifying the name of the dataset columns

In [ ]:
df.rename(columns = lambda column:'_'.join(column.title().split()), inplace = True)

In [ ]:
columns = df.columns.tolist()[:5]
for column in columns:
    df[column] = df[column].apply(lambda name:name.title())

In [ ]:
df.info()

In [ ]:
exams = df.columns.tolist()[-3:]

In [ ]:
df['Math_Score'] = pd.to_numeric(df['Math_Score'],downcast = 'float')
df['Reading_Score'] = pd.to_numeric(df['Reading_Score'], downcast = 'float')
df['Writing_Score'] = pd.to_numeric(df['Writing_Score'], downcast = 'float')

Finding missings

In [ ]:
(df.isna().sum()/df.shape[0])*100

# Descriptive statistics

In [ ]:
for col in df.iloc[:,:5]:
    print(col)
    print(df[col].value_counts())
    print()

In [ ]:
df.describe().T

Descriptive statistics by gender

In [ ]:
gender = (df.groupby('Gender')[exams]
          .agg([np.mean, np.std, np.median]))

In [ ]:
gender.T

In [ ]:
gender_keys = (df.groupby(['Gender','Test_Preparation_Course'])[exams])
metrics = [np.mean, np.std, np.median, min, max]
gender_keys.agg(metrics).T

In [ ]:
(df.groupby(['Race/Ethnicity', 'Parental_Level_Of_Education'])
 [exams]).agg(np.mean)

In [ ]:
df['Total_Score'] = df['Math_Score']+df['Reading_Score']+df['Writing_Score']

# Visualization

In [ ]:
df['Gender'].value_counts().plot(kind = 'bar', figsize = (8, 5))
plt.title('Gender ratio', fontsize= 15)
plt.xlabel('Gender')
plt.xticks([0, 1], ['Female', 'Male'], rotation = 0)
plt.ylabel('Quantity');

In [ ]:
plt.figure(figsize = (8, 5))
df['Race/Ethnicity'].value_counts().sort_index().plot(kind = 'bar')
plt.title('Race/Ethnicity groups')

In [ ]:
plt.figure(figsize = (10, 5))
sns.countplot(
    data = df,
    x = 'Parental_Level_Of_Education',
    hue= 'Gender'
)
plt.title('Parents level of education by gender', pad = 20)
plt.tight_layout()

In [ ]:
plt.figure(figsize = (10, 5))
sns.countplot(
    data = df,
    x = 'Lunch',
    hue = 'Gender'
)
plt.title('Lunch (Standard or Free/Reduced) distribution')
plt.tight_layout()

In [ ]:
df[exams].plot(kind = 'box', figsize = (10,5))
plt.title('Boxplot score');

In [ ]:
fig, (ax0, ax1, ax2) = plt.subplots(ncols = 3, figsize = (15,6))

sns.kdeplot(data = df, x = 'Math_Score', hue = 'Test_Preparation_Course', ax = ax0)
ax0.set_xlabel('Math Score')

sns.kdeplot(data = df, x = 'Reading_Score', hue = 'Test_Preparation_Course', ax = ax1)
ax1.set_xlabel('Reading Score')

sns.kdeplot(data = df, x = 'Writing_Score', hue = 'Test_Preparation_Course', ax = ax2)
ax2.set_xlabel('Writing Score')

plt.tight_layout(pad = 3.5)

In [ ]:
correlation_matrix = df.corr()
mask = np.zeros_like(correlation_matrix)
mask[np.triu_indices_from(mask)] = True
sns.heatmap(
    correlation_matrix,
    annot = True,
    mask = mask,
    square = True,
    cmap = 'Blues');

In [ ]:
sns.pairplot(df, hue = 'Gender', plot_kws = {'alpha':0.5});

In [ ]:
plt.rcParams['axes.facecolor'] = '#EEF1F0'
plt.rcParams['figure.facecolor'] ="#D0D6D4"

fig, (ax0, ax1, ax2) = plt.subplots(ncols = 3, constrained_layout = False, figsize = (10, 5))

sns.violinplot(data = df, y = 'Math_Score', color = 'blue', linewidth=1, ax = ax0)
ax0.set_title('Math Score')

sns.violinplot(data = df, y = 'Reading_Score', color = 'red', linewidth = 1, ax = ax1)
ax1.set_title('Reading Score')

sns.violinplot(data = df, y = 'Writing_Score', color = 'green', linewidth = 1, ax = ax2)
ax2.set_title('Writing Score')

plt.tight_layout(pad = 3.5)


# Machine Learning Model

In [ ]:
from sklearn import metrics
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.preprocessing import OrdinalEncoder
from sklearn.linear_model import (LinearRegression, 
                                  Ridge, 
                                  ElasticNet, 
                                  Lasso)
from sklearn.model_selection import (GridSearchCV, 
                                     train_test_split, 
                                     KFold,
                                     cross_val_score)

### Handling Categorical Data

In [ ]:
df1 = df.copy()

In [ ]:
for col in df1.columns.values[:5]:
    df1[col] = ord_encoder.fit_transform(df1[col].values.reshape(-1,1))

### Building a Linear Regression Model

In [ ]:
x = df1.drop(columns = ['Total_Score', 'Math_Score'], axis = 1)
y = df1['Math_Score']

In [ ]:
x_train, x_test, y_train, y_test = train_test_split(x,y, test_size = 0.3)

In [ ]:
x_train.shape, y_train.shape

In [ ]:
x_test.shape, y_test.shape

Kfold

In [ ]:
kfold()

In [ ]:
kfold = KFold(n_splits = 8)
linear_regression = LinearRegression()
results = cross_val_score(linear_regression, x, y, cv = kfold)

In [ ]:
results.mean(),results

Linear Regression

In [ ]:
linear_regression.fit(x_train, y_train)

In [ ]:
predictions = linear_regression.predict(x_test)

In [ ]:
linear_regression.score(x_test, y_test)

In [ ]:
linear_regression.coef_

Metrics

MAE

In [ ]:
metrics.mean_absolute_error(y_test, predictions)

MSE

In [ ]:
metrics.mean_squared_error(y_test, predictions)

RMSE

In [ ]:
np.sqrt(metrics.mean_squared_error(y_test, predictions))

R2

In [ ]:
metrics.r2_score(y_test, predictions)

## Gradient Boosting

In [ ]:
grad_model = GradientBoostingRegressor(n_estimators = 300, max_depth = 5)

In [ ]:
grad_model.fit(x_train, y_train)

In [ ]:
grad_predictions = grad_model.predict(x_test)

R2

In [ ]:
metrics.r2_score(y_test, grad_predictions)

## Other models

In [ ]:
kfold = KFold(n_splits = 10)
lasso = Lasso()
ridge = Ridge()
elastic_net = ElasticNet()
results_lasso = cross_val_score(lasso, x, y, cv = kfold)
results_ridge = cross_val_score(ridge, x, y, cv = kfold)
results_elastic_net = cross_val_score(elastic_net, x, y, cv = kfold)

In [ ]:
dic_models = {
    'Lasso': results_lasso.mean(),
    'Ridge':results_ridge.mean(),
    'ElasticNet':results_elastic_net.mean()}

In [ ]:
for model, score in dic_models.items():
    print(model,score)